# EE5180 — Seq2Seq (Sutskever et al., 2014) — WMT'14 En→Fr on a Colab GPU

Runs **the same `src/seq2seq` code** as the local setup — nothing is reimplemented here.
Produces the two reported Table 1 rows (single forward vs single reversed LSTM), scored on
**newstest2014, the paper's own ntst14**.

**Why Colab:** a 16 GB Mac shared with other applications leaves the trainer memory-starved
(measured: 91 MB resident when it wants ~890 MB, ~1.2 s/step). A T4 has dedicated VRAM and no
such contention — expect roughly **2–4 h for both arms** instead of ~21 h.

### Before you start
1. **Runtime → Change runtime type → T4 GPU.**
2. Upload `EE5180_seq2seq_submission.zip` (or the whole `seq2seq-ee5180` folder) to your Google Drive.
3. Run the cells in order. Total unattended time ≈ 3–5 h including data preparation.

> Absolute BLEU from these runs is **not** comparable to the paper's Table 1 — see the
> scale-gap statement in the README. What reproduces is the direction and shape of the effects.


## 1. Mount Drive and locate the repository


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Point REPO at the folder containing README.md and src/seq2seq.
# If you uploaded the zip instead of the folder, unzip it first:
#   !mkdir -p /content/seq2seq-ee5180 && unzip -q '/content/drive/MyDrive/EE5180_seq2seq_submission.zip' -d /content/seq2seq-ee5180
#   REPO = '/content/seq2seq-ee5180'
REPO = '/content/drive/MyDrive/seq2seq-ee5180'   # <-- adjust to where you put it

import os, sys
assert os.path.exists(os.path.join(REPO, 'src', 'seq2seq', 'train.py')), f'not a repo: {REPO}'
os.chdir(REPO); sys.path.insert(0, os.path.join(REPO, 'src'))
print('repo:', REPO)


## 2. Dependencies
Colab ships torch built for CUDA — **do not** reinstall it from `requirements.txt`.


In [ ]:
!pip -q install sacrebleu==2.6.0 sacremoses==0.2.0 pyyaml tqdm
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY - set the T4 runtime!')


## 3. Correctness gates
Same suite as locally (20 tests). The MPS/CPU test skips on CUDA; everything else must pass.


In [ ]:
!python -m pytest tests/test_smoke.py -q


## 4. Work directory
`EE5180_WORK` holds corpora and checkpoints. Put it on **Drive** so a disconnect doesn't lose
the run — training checkpoints every epoch and resumes from `last.pt` automatically.


In [ ]:
os.environ['EE5180_WORK'] = '/content/drive/MyDrive/ee5180-work'
!mkdir -p "$EE5180_WORK"
print('work dir:', os.environ['EE5180_WORK'])


## 5. Data — WMT'14 En→Fr
News-Commentary v9 + Europarl v7 (both official WMT'14 training corpora), dev = newstest2013,
test = **newstest2014 full 3003 sentences = the paper's ntst14**, fetched through sacreBLEU.

~700 MB download; tokenisation of 1.28M pairs takes a few minutes. Skips work already done.


In [ ]:
!./scripts/get_wmt14.sh


In [ ]:
!python scripts/subsample_parallel.py \
  --src-in  "$EE5180_WORK/data/wmt14/raw/europarl.en" \
  --tgt-in  "$EE5180_WORK/data/wmt14/raw/europarl.fr" \
  --src-out "$EE5180_WORK/data/wmt14/raw/europarl.sub.en" \
  --tgt-out "$EE5180_WORK/data/wmt14/raw/europarl.sub.fr" \
  --n 1100000 --seed 1


In [ ]:
!python scripts/prepare_data.py --out "$EE5180_WORK/data/wmt14/prepared" \
  --train-src "$EE5180_WORK/data/wmt14/raw/nc9.en" "$EE5180_WORK/data/wmt14/raw/europarl.sub.en" \
  --train-tgt "$EE5180_WORK/data/wmt14/raw/nc9.fr" "$EE5180_WORK/data/wmt14/raw/europarl.sub.fr" \
  --dev-src  "$EE5180_WORK/data/wmt14/raw/newstest2013.en" \
  --dev-tgt  "$EE5180_WORK/data/wmt14/raw/newstest2013.fr" \
  --test-src "$EE5180_WORK/data/wmt14/raw/newstest2014.en" \
  --test-tgt "$EE5180_WORK/data/wmt14/raw/newstest2014.fr" \
  --src-vocab-size 32000 --tgt-vocab-size 32000 --max-train-pairs 500000 --workers 2 --seed 1


In [ ]:
# Sanity: test must be exactly 3003 lines (the paper's ntst14)
!wc -l "$EE5180_WORK/data/wmt14/raw/newstest2014.en"
!ls -la "$EE5180_WORK/data/wmt14/prepared"/*.npz


## 6. Size the run for this GPU
Prints target words/s and projected hours per epoch, so you know what you're committing to.


In [ ]:
!python scripts/benchmark.py --corpus-pairs 500000


## 7. Train the two reported rows
The reversal flag is the **only** difference between them: same data, same seed, same batch
order, same schedule. Each cell resumes from its own last checkpoint if interrupted —
just re-run the cell after a disconnect.

**Run these one at a time.** Two concurrent trainings contend and both get slower.


In [ ]:
!PYTHONPATH=src python -m seq2seq.train --config configs/wmt14_small.yaml --reverse-source --seed 1


In [ ]:
!PYTHONPATH=src python -m seq2seq.train --config configs/wmt14_small.yaml --forward-source --seed 1


## 8. Decode, score and plot
Beam 1, 2 and 12 for each arm, scored two ways (tokenised = the `multi-bleu.pl` equivalent the
paper used; plus standard sacreBLEU on detokenised output). Writes `results/wmt14_small/`.

Decoding is deliberately **not** run while training — that contention is what crippled the
local attempt.


In [ ]:
!python scripts/make_results.py --name wmt14_small \
  --data-dir "$EE5180_WORK/data/wmt14/prepared" \
  --runs-root "$EE5180_WORK/runs/wmt14" \
  --out results/wmt14_small --beams 1 2 12 --seeds 1 --device cuda \
  --scale-note "0.5M training pairs vs the paper's 12M; 2x512 vs 4x1000; 32k/32k vocab vs 160k/80k. Absolute BLEU is NOT comparable to Table 1 - the direction and shape of the effects are."

print(open('results/wmt14_small/results.md').read())


## 9. Rebuild the report
Reads `results/*/all_results.json`, so the prose cannot drift from the runs.


In [ ]:
!python scripts/make_report.py
print(open('report/EE5180_midterm_report.md').read()[:3000])


## 10. Bring the results home
If `REPO` is on Drive, `results/` and `report/` are already synced — just pull them down on the Mac.
Otherwise download the archive this cell writes, unzip it over the local repo, then run:

```bash
make report    # regenerates the PDF (needs Chrome, which Colab lacks)
make slides    # regenerates the .pptx (needs node)
```


In [ ]:
!cd "$REPO" && zip -qr /content/ee5180_results.zip results report && ls -la /content/ee5180_results.zip
from google.colab import files
files.download('/content/ee5180_results.zip')


## 11. Optional stretch — seeds 2–3 and the ensemble rows
Cheap on a GPU, and it makes the Table 1 ensemble/beam trends visible alongside the two
required rows.


In [ ]:
for seed in (2, 3):
    !PYTHONPATH=src python -m seq2seq.train --config configs/wmt14_small.yaml --reverse-source --seed {seed}
    !PYTHONPATH=src python -m seq2seq.train --config configs/wmt14_small.yaml --forward-source --seed {seed}


In [ ]:
!python scripts/make_results.py --name wmt14_small \
  --data-dir "$EE5180_WORK/data/wmt14/prepared" \
  --runs-root "$EE5180_WORK/runs/wmt14" \
  --out results/wmt14_small --beams 1 2 12 --seeds 1 2 3 --ensemble --device cuda \
  --scale-note "0.5M training pairs vs the paper's 12M; 2x512 vs 4x1000; 32k/32k vocab vs 160k/80k. Absolute BLEU is NOT comparable to Table 1."
